In [ ]:
# ============================================================
# ANALYSE DE LA BDD : RISQUES D'AUDIT PAR CYCLE COMPTABLE
# Compatible Google Colab
# ============================================================

# =========================
# 1) INSTALLATION / IMPORTS
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# =========================
# 2) IMPORTER LE FICHIER
# =========================
print("Upload ton fichier Excel : base_donnees_risques_audit_cycle_comptable.xlsx")
uploaded = files.upload()

# récupérer automatiquement le nom du fichier uploadé
file_name = list(uploaded.keys())[0]
print(f"Fichier chargé : {file_name}")

# =========================
# 3) LECTURE DE LA FEUILLE
# =========================
df = pd.read_excel(file_name, sheet_name="Audit_Risks_DB")

print("\nAperçu des données :")
display(df.head())

print("\nDimensions de la base :")
print(f"Lignes : {df.shape[0]}")
print(f"Colonnes : {df.shape[1]}")

print("\nColonnes disponibles :")
print(df.columns.tolist())

# =========================
# 4) NETTOYAGE DES DONNÉES
# =========================
# supprimer les espaces inutiles dans les noms de colonnes
df.columns = df.columns.str.strip()

# supprimer les espaces inutiles dans les colonnes texte
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# remplacer les valeurs 'nan' textuelles éventuelles par NaN réel
df = df.replace(["nan", "None", ""], np.nan)

print("\nValeurs manquantes par colonne :")
display(df.isna().sum().to_frame("Nb_valeurs_manquantes"))

# doublons
nb_doublons = df.duplicated().sum()
print(f"\nNombre de doublons exacts : {nb_doublons}")

# supprimer doublons si besoin
df_clean = df.drop_duplicates().copy()

print("\nDimensions après suppression des doublons :")
print(df_clean.shape)

# =========================
# 5) CONTRÔLES DE QUALITÉ
# =========================
print("\nTypes de données :")
print(df_clean.dtypes)

print("\nNombre de valeurs uniques par colonne principale :")
for col in [
    "Cycle comptable",
    "Processus",
    "Type de risque",
    "Assertion impactée",
    "Niveau de risque"
]:
    if col in df_clean.columns:
        print(f"{col}: {df_clean[col].nunique()}")

# =========================
# 6) STATISTIQUES DESCRIPTIVES
# =========================
print("\n--- Répartition par cycle comptable ---")
cycle_counts = df_clean["Cycle comptable"].value_counts()
display(cycle_counts.to_frame("Nombre de risques"))

print("\n--- Répartition par type de risque ---")
type_counts = df_clean["Type de risque"].value_counts()
display(type_counts.to_frame("Nombre de risques"))

print("\n--- Répartition par assertion impactée ---")
assertion_counts = df_clean["Assertion impactée"].value_counts()
display(assertion_counts.to_frame("Nombre de risques"))

print("\n--- Répartition par niveau de risque ---")
niveau_counts = df_clean["Niveau de risque"].value_counts()
display(niveau_counts.to_frame("Nombre de risques"))

# =========================
# 7) VISUALISATIONS
# =========================

# --- Graphique 1 : risques par cycle comptable
plt.figure()
cycle_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("Nombre de risques par cycle comptable")
plt.xlabel("Cycle comptable")
plt.ylabel("Nombre de risques")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --- Graphique 2 : risques par type
plt.figure()
type_counts.plot(kind="bar")
plt.title("Répartition des risques par type")
plt.xlabel("Type de risque")
plt.ylabel("Nombre de risques")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# --- Graphique 3 : risques par assertion
plt.figure()
assertion_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("Répartition des risques par assertion impactée")
plt.xlabel("Assertion")
plt.ylabel("Nombre de risques")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --- Graphique 4 : risques par niveau
plt.figure()
niveau_counts.reindex(["Faible", "Moyen", "Élevé"]).dropna().plot(kind="bar")
plt.title("Répartition des risques par niveau")
plt.xlabel("Niveau de risque")
plt.ylabel("Nombre de risques")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# =========================
# 8) TABLEAUX CROISÉS DYNAMIQUES
# =========================

# tableau croisé : cycle x niveau de risque
pivot_cycle_niveau = pd.crosstab(
    df_clean["Cycle comptable"],
    df_clean["Niveau de risque"],
    margins=True
)

print("\n--- Tableau croisé : Cycle comptable x Niveau de risque ---")
display(pivot_cycle_niveau)

# tableau croisé : cycle x type de risque
pivot_cycle_type = pd.crosstab(
    df_clean["Cycle comptable"],
    df_clean["Type de risque"],
    margins=True
)

print("\n--- Tableau croisé : Cycle comptable x Type de risque ---")
display(pivot_cycle_type)

# tableau croisé : assertion x niveau
pivot_assertion_niveau = pd.crosstab(
    df_clean["Assertion impactée"],
    df_clean["Niveau de risque"],
    margins=True
)

print("\n--- Tableau croisé : Assertion x Niveau de risque ---")
display(pivot_assertion_niveau)

# =========================
# 9) SCORE NUMÉRIQUE DES RISQUES
# =========================
# On transforme le niveau en score pour faire des analyses quantitatives
mapping_score = {
    "Faible": 1,
    "Moyen": 2,
    "Élevé": 3
}

df_clean["Score risque"] = df_clean["Niveau de risque"].map(mapping_score)

print("\nAperçu avec score :")
display(df_clean.head())

# score moyen par cycle
score_moyen_cycle = (
    df_clean.groupby("Cycle comptable")["Score risque"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("\n--- Score moyen de risque par cycle ---")
display(score_moyen_cycle.to_frame("Score_moyen"))

plt.figure()
score_moyen_cycle.plot(kind="bar")
plt.title("Score moyen de risque par cycle comptable")
plt.xlabel("Cycle comptable")
plt.ylabel("Score moyen")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# =========================
# 10) MATRICE DE RISQUES
# =========================
# matrice = nombre de risques par cycle et assertion
matrice_risques = pd.crosstab(
    df_clean["Cycle comptable"],
    df_clean["Assertion impactée"]
)

print("\n--- Matrice des risques : Cycle x Assertion ---")
display(matrice_risques)

# heatmap simple sans seaborn
plt.figure(figsize=(12, 7))
plt.imshow(matrice_risques, aspect="auto")
plt.colorbar(label="Nombre de risques")
plt.xticks(range(len(matrice_risques.columns)), matrice_risques.columns, rotation=45, ha="right")
plt.yticks(range(len(matrice_risques.index)), matrice_risques.index)
plt.title("Heatmap - Matrice des risques (Cycle x Assertion)")
plt.tight_layout()
plt.show()

# =========================
# 11) ANALYSE CIBLÉE : RISQUES DE FRAUDE
# =========================
fraude_df = df_clean[df_clean["Type de risque"].str.lower() == "fraude"]

print("\n--- Risques de fraude ---")
print(f"Nombre total de risques de fraude : {fraude_df.shape[0]}")
display(fraude_df.head(10))

fraude_par_cycle = fraude_df["Cycle comptable"].value_counts()

plt.figure()
fraude_par_cycle.plot(kind="bar")
plt.title("Nombre de risques de fraude par cycle")
plt.xlabel("Cycle comptable")
plt.ylabel("Nombre de risques de fraude")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# =========================
# 12) TOP PROCESSUS LES PLUS EXPOSÉS
# =========================
top_processus = df_clean["Processus"].value_counts().head(10)

print("\n--- Top 10 des processus les plus exposés ---")
display(top_processus.to_frame("Nombre de risques"))

plt.figure()
top_processus.sort_values().plot(kind="barh")
plt.title("Top 10 des processus les plus exposés")
plt.xlabel("Nombre de risques")
plt.ylabel("Processus")
plt.tight_layout()
plt.show()

# =========================
# 13) FILTRAGE DES RISQUES ÉLEVÉS
# =========================
risques_eleves = df_clean[df_clean["Niveau de risque"].str.lower() == "élevé"]

print("\n--- Risques élevés ---")
print(f"Nombre de risques élevés : {risques_eleves.shape[0]}")
display(risques_eleves.head(20))

# risques élevés par cycle
eleves_par_cycle = risques_eleves["Cycle comptable"].value_counts()

plt.figure()
eleves_par_cycle.plot(kind="bar")
plt.title("Risques élevés par cycle comptable")
plt.xlabel("Cycle comptable")
plt.ylabel("Nombre de risques élevés")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# =========================
# 14) RECOMMANDATIONS AUTOMATIQUES
# =========================
print("\n--- Recommandations automatiques ---")

cycle_max_risque = score_moyen_cycle.idxmax()
score_max = score_moyen_cycle.max()

print(f"Cycle le plus risqué selon le score moyen : {cycle_max_risque} (score = {score_max})")

assertion_plus_frequente = df_clean["Assertion impactée"].value_counts().idxmax()
print(f"Assertion la plus exposée : {assertion_plus_frequente}")

type_plus_frequent = df_clean["Type de risque"].value_counts().idxmax()
print(f"Type de risque dominant : {type_plus_frequent}")

print("\nInterprétation possible :")
print("- Prioriser les travaux d'audit sur les cycles ayant le score moyen le plus élevé.")
print("- Renforcer les tests sur les assertions les plus fréquentes.")
print("- Mettre l'accent sur les risques de fraude et les risques élevés.")

# =========================
# 15) EXPORT DES RÉSULTATS
# =========================
# exporter les datasets et tableaux d'analyse dans un nouveau fichier Excel
output_file = "analyse_risques_audit_resultats.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="BDD_nettoyee", index=False)
    cycle_counts.to_frame("Nombre de risques").to_excel(writer, sheet_name="Risques_par_cycle")
    type_counts.to_frame("Nombre de risques").to_excel(writer, sheet_name="Risques_par_type")
    assertion_counts.to_frame("Nombre de risques").to_excel(writer, sheet_name="Risques_par_assertion")
    niveau_counts.to_frame("Nombre de risques").to_excel(writer, sheet_name="Risques_par_niveau")
    pivot_cycle_niveau.to_excel(writer, sheet_name="Pivot_cycle_niveau")
    pivot_cycle_type.to_excel(writer, sheet_name="Pivot_cycle_type")
    pivot_assertion_niveau.to_excel(writer, sheet_name="Pivot_assertion_niveau")
    matrice_risques.to_excel(writer, sheet_name="Matrice_cycle_assertion")
    score_moyen_cycle.to_frame("Score_moyen").to_excel(writer, sheet_name="Score_moyen_cycle")
    risques_eleves.to_excel(writer, sheet_name="Risques_eleves", index=False)
    fraude_df.to_excel(writer, sheet_name="Risques_fraude", index=False)

print(f"\nFichier exporté : {output_file}")

# téléchargement du fichier résultat
files.download(output_file)